In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Week 6") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.2.0


In [7]:
data = [
    (1, "Manas", 21),
    (2, "Manas 2", 22),
    (3, "Manas 3", 20)
]

df = spark.createDataFrame(data, ["ID", "Name", "Age"])

df.show()

+---+-------+---+
| ID|   Name|Age|
+---+-------+---+
|  1|  Manas| 21|
|  2|Manas 2| 22|
|  3|Manas 3| 20|
+---+-------+---+



 
Q1: Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

ANSWER 1:

Answer

Apache Spark follows a distributed architecture where different components work together to process large amounts of data efficiently. The three main components are Driver, Cluster Manager, and Executors.

Driver

The Driver is the main program of a Spark application. It creates the SparkSession, reads the user's Spark code, converts it into tasks, and coordinates the execution. It also collects the results from the Executors and displays the final output.

Cluster Manager

The Cluster Manager is responsible for managing the resources of the cluster. It allocates CPU and memory, starts Executor processes on worker nodes, and ensures that the application has the resources required for execution. Spark supports Cluster Managers such as Standalone, YARN, Kubernetes, and Mesos.

Executor

Executors are worker processes that perform the actual computation. They execute the tasks assigned by the Driver, process the data, store intermediate results in memory if needed, and send the final results back to the Driver.

Q2: How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?


Lazy Evaluation is one of Spark's most powerful features for performance optimization. Instead of executing transformations immediately, Spark records them and builds an execution plan called a Directed Acyclic Graph (DAG). The actual computation only begins when you call an Action like show(), count(), or collect().

This delay gives Spark time to analyze all operations together, combine multiple transformations, eliminate redundant steps, and minimize data movement between nodes. The result: faster execution and smarter resource usage.

In this below example, filter() and select() are transformations—Spark just records them in the DAG. Nothing actually executes until collect() is called, which is an Action. At that point, Spark optimizes the entire plan and processes everything in one efficient pass, returning the results as a list.

df = spark.read.csv("data/source.csv", header=True, inferSchema=True)

filtered_df = df.filter(df.category == "Electronics")
selected_df = filtered_df.select("product_name", "price")

selected_df.show()

Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.


In [4]:
csv_file_path = "./data_week_6/SampleSuperstore.csv"

df = spark.read.csv( csv_file_path, header=True, inferSchema=True)

df.show(5)

+--------------+---------+-------------+---------------+----------+-----------+------+---------------+------------+--------+--------+--------+--------+
|     Ship Mode|  Segment|      Country|           City|     State|Postal Code|Region|       Category|Sub-Category|   Sales|Quantity|Discount|  Profit|
+--------------+---------+-------------+---------------+----------+-----------+------+---------------+------------+--------+--------+--------+--------+
|  Second Class| Consumer|United States|      Henderson|  Kentucky|      42420| South|      Furniture|   Bookcases|  261.96|       2|     0.0| 41.9136|
|  Second Class| Consumer|United States|      Henderson|  Kentucky|      42420| South|      Furniture|      Chairs|  731.94|       3|     0.0| 219.582|
|  Second Class|Corporate|United States|    Los Angeles|California|      90036|  West|Office Supplies|      Labels|   14.62|       2|     0.0|  6.8714|
|Standard Class| Consumer|United States|Fort Lauderdale|   Florida|      33311| South|  

In [8]:
df.printSchema()

root
 |-- Ship Mode: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Q4: What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

Answer 4:

CSV and Parquet are two commonly used file formats in Apache Spark.

A CSV file stores data in a plain text, row-wise format, making it easy to read and edit. However, it occupies more storage space and Spark has to infer the schema every time unless it is specified manually.

Parquet is a columnar file format that stores data in a compressed binary form. Since it stores the schema along with the data, Spark can read it more efficiently. It also reads only the required columns instead of the entire dataset, which improves performance and reduces processing time.

For example, if a dataset contains 20 columns but only the Sales and Profit columns are needed, Spark reads all 20 columns from a CSV file, whereas it reads only the required columns from a Parquet file. This makes Parquet a better choice for large-scale data processing.

In [5]:
data = [
    (1, "Laptop", "Electronics", 75000, "Completed", 80000, 5000, "North"),
    (2, "Smartphone", "Electronics", 30000, "Pending", 32000, 2000, "South"),
    (3, "Headphones", "Electronics", 2500, "Completed", 3000, 500, "East"),
    (4, "Office Chair", "Furniture", 8500, "Completed", 9000, 500, "West"),
    (5, "Study Table", "Furniture", 12000, "Pending", 13000, 1000, "North"),
    (6, "Refrigerator", "Appliances", 45000, "Completed", 47000, 2000, "South"),
    (7, "Microwave", "Appliances", 9000, "Cancelled", 9500, 500, "East"),
    (8, "Smart Watch", "Electronics", 15000, "Completed", 17000, 2000, "West"),
    (9, "Printer", "Electronics", 18000, "Pending", 20000, 2000, "North"),
    (10, "Bookshelf", "Furniture", 7000, "Completed", 7500, 500, "South")
]

columns = [
    "product_id",
    "product_name",
    "category",
    "price",
    "status",
    "base_price",
    "discount",
    "region"
]

df = spark.createDataFrame(data, columns)

df.show()

+----------+------------+-----------+-----+---------+----------+--------+------+
|product_id|product_name|   category|price|   status|base_price|discount|region|
+----------+------------+-----------+-----+---------+----------+--------+------+
|         1|      Laptop|Electronics|75000|Completed|     80000|    5000| North|
|         2|  Smartphone|Electronics|30000|  Pending|     32000|    2000| South|
|         3|  Headphones|Electronics| 2500|Completed|      3000|     500|  East|
|         4|Office Chair|  Furniture| 8500|Completed|      9000|     500|  West|
|         5| Study Table|  Furniture|12000|  Pending|     13000|    1000| North|
|         6|Refrigerator| Appliances|45000|Completed|     47000|    2000| South|
|         7|   Microwave| Appliances| 9000|Cancelled|      9500|     500|  East|
|         8| Smart Watch|Electronics|15000|Completed|     17000|    2000|  West|
|         9|     Printer|Electronics|18000|  Pending|     20000|    2000| North|
|        10|   Bookshelf|  F

In [19]:
from pyspark.sql.functions import col
new_df = df.filter(df.category == "Electronics").select("product_name", "price")

new_df.show()

+------------+-----+
|product_name|price|
+------------+-----+
|      Laptop|75000|
|  Smartphone|30000|
|  Headphones| 2500|
| Smart Watch|15000|
|     Printer|18000|
+------------+-----+



Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.


In [20]:
from pyspark.sql.functions import col

new_df = (
    df.withColumnRenamed("product_name", "item_name")
      .withColumn("price", col("price").cast("double"))
)

new_df.show()

+----------+------------+-----------+-------+---------+----------+--------+------+
|product_id|   item_name|   category|  price|   status|base_price|discount|region|
+----------+------------+-----------+-------+---------+----------+--------+------+
|         1|      Laptop|Electronics|75000.0|Completed|     80000|    5000| North|
|         2|  Smartphone|Electronics|30000.0|  Pending|     32000|    2000| South|
|         3|  Headphones|Electronics| 2500.0|Completed|      3000|     500|  East|
|         4|Office Chair|  Furniture| 8500.0|Completed|      9000|     500|  West|
|         5| Study Table|  Furniture|12000.0|  Pending|     13000|    1000| North|
|         6|Refrigerator| Appliances|45000.0|Completed|     47000|    2000| South|
|         7|   Microwave| Appliances| 9000.0|Cancelled|      9500|     500|  East|
|         8| Smart Watch|Electronics|15000.0|Completed|     17000|    2000|  West|
|         9|     Printer|Electronics|18000.0|  Pending|     20000|    2000| North|
|   

Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Answer 7:

Spark uses a Lineage Graph, also called a Directed Acyclic Graph (DAG), to keep track of all the transformations performed on a dataset. Instead of saving multiple copies of the data, Spark remembers how the data was created.

If a worker node fails and some data is lost, Spark checks the Lineage Graph to find the sequence of operations that produced the lost data. It then re-executes only those transformations to recreate the missing partitions instead of processing the entire dataset again. This helps Spark recover quickly and provides fault tolerance.

Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [22]:
from pyspark.sql.functions import col

new_df = df.filter(
    (col("status") == "Completed") & (col("price") > 1000)
)

new_df.show()

+----------+------------+-----------+-----+---------+----------+--------+------+
|product_id|product_name|   category|price|   status|base_price|discount|region|
+----------+------------+-----------+-----+---------+----------+--------+------+
|         1|      Laptop|Electronics|75000|Completed|     80000|    5000| North|
|         3|  Headphones|Electronics| 2500|Completed|      3000|     500|  East|
|         4|Office Chair|  Furniture| 8500|Completed|      9000|     500|  West|
|         6|Refrigerator| Appliances|45000|Completed|     47000|    2000| South|
|         8| Smart Watch|Electronics|15000|Completed|     17000|    2000|  West|
|        10|   Bookshelf|  Furniture| 7000|Completed|      7500|     500| South|
+----------+------------+-----------+-----+---------+----------+--------+------+



Q9: Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Answer 9:

Predicate Pushdown is an optimization technique used by Spark when reading Parquet files. Instead of loading the entire dataset into memory, Spark pushes the filtering condition down to the Parquet file itself. As a result, only the rows that satisfy the condition are read into memory.

This reduces the amount of data that needs to be read from disk, decreases memory usage, and improves query performance, especially when working with large datasets.

Example

Suppose a Parquet file contains data for 1 million customers, but we only need customers from the North region.

df.filter(df.region == "North")

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [23]:
from pyspark.sql.functions import col

new_df = df.withColumn("final_price", col("base_price") * 1.18)

new_df.show()

+----------+------------+-----------+-----+---------+----------+--------+------+-----------+
|product_id|product_name|   category|price|   status|base_price|discount|region|final_price|
+----------+------------+-----------+-----+---------+----------+--------+------+-----------+
|         1|      Laptop|Electronics|75000|Completed|     80000|    5000| North|    94400.0|
|         2|  Smartphone|Electronics|30000|  Pending|     32000|    2000| South|    37760.0|
|         3|  Headphones|Electronics| 2500|Completed|      3000|     500|  East|     3540.0|
|         4|Office Chair|  Furniture| 8500|Completed|      9000|     500|  West|    10620.0|
|         5| Study Table|  Furniture|12000|  Pending|     13000|    1000| North|    15340.0|
|         6|Refrigerator| Appliances|45000|Completed|     47000|    2000| South|    55460.0|
|         7|   Microwave| Appliances| 9000|Cancelled|      9500|     500|  East|    11210.0|
|         8| Smart Watch|Electronics|15000|Completed|     17000|    20

Q11: What is the difference between Transformations and Actions? Provide two examples of each.

Answer

In Apache Spark, operations are divided into Transformations and Actions.

Transformations are operations that create a new DataFrame or RDD from an existing one. They are lazy, meaning Spark does not execute them immediately. Instead, it records them and executes them only when an action is called.

Actions are operations that trigger the execution of all pending transformations. They either return a result to the driver or write the data to storage.

Transformations are :

filter() – Selects rows that satisfy a given condition.
select() – Selects specific columns from a DataFrame.

Actions re:

show() – Displays the DataFrame on the screen.

collect() – Returns all the data from the DataFrame to the driver program.

Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [ ]:
df = spark.read.parquet("./data_week_6/sample.parquet")

df = df.filter(df.user_id.isNotNull())

df.write.csv("./data_week_6/output", header=True)b

Q13: In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

Answer 13:

In Apache Spark, applications can run in Client Mode or Cluster Mode depending on where the Driver program is executed.

In Client Mode, the Driver runs on the user's local machine, while the executors run on the cluster. The application depends on the client machine remaining connected. If the client machine shuts down or loses connection, the Spark application may stop.

In Cluster Mode, the Driver runs inside the cluster along with the executors. The cluster manages the entire application, so it can continue running even if the client disconnects. This mode is more suitable for long-running and production applications.

Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

In [7]:
from pyspark.sql.functions import col

data = [
    (1, "Laptop", "Electronics", 75000, "Completed", 80000, 5000, "North", "High"),
    (2, "Smartphone", "Electronics", 30000, "Pending", 32000, 2000, "South", "Medium"),
    (3, "Headphones", "Electronics", 2500, "Completed", 3000, 500, "East", "Low"),
    (4, "Office Chair", "Furniture", 8500, "Completed", 9000, 500, "West", "High")
]

columns = [
    "product_id",
    "product_name",
    "category",
    "price",
    "status",
    "base_price",
    "discount",
    "region",
    "priority"
]

df = spark.createDataFrame(data, columns)

new_df = df.filter(
    (col("region") == "North") | (col("priority") == "High")
)

new_df.show()

+----------+------------+-----------+-----+---------+----------+--------+------+--------+
|product_id|product_name|   category|price|   status|base_price|discount|region|priority|
+----------+------------+-----------+-----+---------+----------+--------+------+--------+
|         1|      Laptop|Electronics|75000|Completed|     80000|    5000| North|    High|
|         4|Office Chair|  Furniture| 8500|Completed|      9000|     500|  West|    High|
+----------+------------+-----------+-----+---------+----------+--------+------+--------+



Q15: When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

Answer

It is safer to use .show(5) because it displays only the first 5 rows of the dataset. Spark retrieves only a small amount of data, making it fast and memory-efficient.

On the other hand, 
.collect() retrieves all the rows from the dataset and stores them in the Driver's memory. For a multi-terabyte dataset, this can consume a huge amount of memory, causing the application to become slow or even crash with an Out of Memory (OOM) error.